In [2]:
! python ingest.py

Loading Markdown files...
Loaded 6 documents
Chunking documents...
Created 68 chunks
Preparing text for embeddings...
Prepared 68 chunks for embedding


#  Embedding

In [3]:
import json
from pathlib import Path

from sentence_transformers import SentenceTransformer

In [4]:
INPUT_FILE = Path("processed_chunks.json")
OUTPUT_FILE = Path("chunks_with_embeddings.json")

MODEL_NAME = "all-MiniLM-L6-v2"


In [5]:
print(f"Loading embedding model: {MODEL_NAME}")

model = SentenceTransformer(MODEL_NAME)

print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading embedding model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


/tmp/ipykernel_7249/319083760.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [6]:
print(f"Loading {INPUT_FILE}...")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

Loading processed_chunks.json...
Loaded 68 chunks


In [7]:

def build_embedding_text(chunk):
    """
    Build the text sent to the embedding model.

    We include document title and section headers so the
    embedding contains additional context.
    """

    parts = [
        chunk.get("title"),
        chunk.get("header1"),
        chunk.get("header2"),
        chunk.get("header3"),
        chunk.get("text"),
    ]

    return "\n".join(
        part.strip()
        for part in parts
        if part
    )

In [8]:
texts = [build_embedding_text(chunk) for chunk in chunks]

In [9]:
print("Generating embeddings...")

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print("Embeddings generated.")

Generating embeddings...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings generated.


In [10]:
for chunk, embedding in zip(chunks, embeddings):
    chunk["embedding"] = embedding.tolist()

In [11]:
print(f"Saving {OUTPUT_FILE}...")

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        chunks,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Done!")

Saving chunks_with_embeddings.json...
Done!


### minsearch

In [15]:
import json
from pathlib import Path

from text_search import Index

Loading processed_chunks.json...
Loaded 68 chunks
Building MinSearch index...
Index ready.


In [17]:
! python text_search.py

Loading processed_chunks.json...
Loaded 68 chunks
Building MinSearch index...
Index ready.

Query (or 'exit'): Traceback (most recent call last):
  File "/workspaces/DevDocs-AI-RAG-Chat/text_search.py", line 86, in <module>
    query = input("\nQuery (or 'exit'): ")
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
^C
